# SAFEWORK AI: latih model APD v2

Notebook ini mengerjakan semuanya dalam satu kali **Run All** di Colab (runtime GPU T4):

1. Unduh dua dataset publik (tanpa API key) dan gabungkan menjadi 14 kelas: helmet, gloves, vest, boots, goggles, Person, mask, serta pasangan `no_*`.
2. Lengkapi label yang tidak ada di dataset kedua (Person, vest, no_vest, forklift) dengan model, supaya model tidak belajar bahwa orang dan rompi adalah latar belakang.
3. Latih YOLO11n, lalu bandingkan dengan model baseline `ppe-v1` pada 200 gambar validasi Construction-PPE yang sama.

| Sumber | Isi | Lisensi |
|---|---|---|
| Ultralytics Construction-PPE (rilis 3 Nov 2026) | 1.000 gambar, 12 kelas termasuk `no_*` | CC BY 4.0 |
| PPEs Dataset, Roboflow Universe (salinan Hugging Face `keremberke/protective-equipment-detection`) | 11.978 gambar: glove, goggles, helmet, mask, shoes dan pasangan `no_*` | CC BY 4.0 |

Perkiraan waktu: sekitar 2 sampai 3 jam (kasar, bergantung antrean CPU Colab). Bila sesi putus, jalankan ulang semua sel; checkpoint di Google Drive otomatis dilanjutkan.

Hasil: `best.pt` disalin ke Drive (`safework_ppe_v2/`) dan ke `/content/out/`.

Wajib: cantumkan kedua sumber di README atau dokumen lomba (atribusi CC BY 4.0).

In [1]:
# 1) Persiapan: paket, GPU/CPU, dan lokasi hasil
import os, sys, re, json, shutil, random, zipfile, subprocess, collections
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics'], check=True)

import torch
import ultralytics

COLAB = False
try:
    import google.colab  # type: ignore
    COLAB = True
except ImportError:
    pass

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = '0' if CUDA_AVAILABLE else 'cpu'
print('ultralytics', ultralytics.__version__)
print('Device:', 'GPU CUDA' if CUDA_AVAILABLE else 'CPU')

REPO = Path.cwd().resolve()
if COLAB:
    REPO = Path('/content/safework')
    if not (REPO / 'pyproject.toml').exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/wildnrfm/safework.git', str(REPO)], check=True)
else:
    while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
        REPO = REPO.parent
    if not (REPO / 'pyproject.toml').exists():
        raise RuntimeError('Jalankan notebook dari folder repository safework-ai')

os.environ['WANDB_MODE'] = 'disabled'
for key in ('wandb', 'clearml', 'comet', 'mlflow', 'neptune', 'tensorboard'):
    try:
        ultralytics.settings.update({key: False})
    except Exception:
        pass

USE_DRIVE = False
DRIVE_DIR = None
if COLAB:
    USE_DRIVE = True
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_DIR = Path('/content/drive/MyDrive/safework_ppe_v2')
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        DRIVE_DIR = None
        print('Drive tidak terhubung:', e)

if COLAB:
    WORK = Path('/content/work')
    RUNS = (DRIVE_DIR / 'runs') if DRIVE_DIR else Path('/content/runs')
else:
    WORK = REPO / 'data_local' / 'training'
    RUNS = REPO / 'runs'

WORK.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)
print('Lokasi data:', WORK)
print('Lokasi hasil:', RUNS)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
ultralytics 8.4.216
Device: GPU CUDA
Mounted at /content/drive
Repository: /content/safework
Lokasi data: /content/work
Lokasi hasil: /content/drive/MyDrive/safework_ppe_v2/runs


In [2]:
# 2) Pengaturan: ubah di sini saja
MODELS = ['yolo11n.pt']
EPOCHS = 60
IMGSZ = 640
BATCH = 32 if CUDA_AVAILABLE else 8
PATIENCE = 20
CACHE = 'disk'
SEED = 0
EXTRA_TRAIN_ARGS = {}

CPPE_OVERSAMPLE = 3
PPES_VAL_SUBSET = 600
PPES_VALID_TO_TRAIN = False
PSEUDO_CONF = 0.5

BASELINE_WEIGHTS = None
REPO_URL = 'https://github.com/wildnrfm/safework.git'
CPPE_URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip'
HF_BASE = 'https://huggingface.co/datasets/keremberke/protective-equipment-detection/resolve/main/data/'

CLASSES = ['helmet', 'gloves', 'vest', 'boots', 'goggles', 'Person', 'no_helmet', 'no_goggle',
           'no_gloves', 'no_boots', 'forklift', 'no_vest', 'mask', 'no_mask']
NAME2ID = {n: i for i, n in enumerate(CLASSES)}
N_CPPE = 12

PPES_MAP = {'glove': 'gloves', 'goggles': 'goggles', 'helmet': 'helmet', 'mask': 'mask',
            'no_glove': 'no_gloves', 'no_goggles': 'no_goggle', 'no_helmet': 'no_helmet',
            'no_mask': 'no_mask', 'no_shoes': 'no_boots', 'shoes': 'boots'}


In [3]:
# 3) Fungsi bantu (hanya definisi; belum ada yang dijalankan)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def norm(name):
    return re.sub(r"[\s\-]+", "_", str(name).strip().lower())


def safe_name(name):
    return re.sub(r"[^A-Za-z0-9_.\-]", "_", str(name))


def download(url, dest):
    import requests
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print("sudah ada:", dest.name)
        return dest
    tmp = dest.with_name(dest.name + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    tmp.rename(dest)
    print(f"selesai: {dest.name} ({dest.stat().st_size / 1e6:.0f} MB)")
    return dest


def unzip(zpath, out_dir):
    out_dir = Path(out_dir)
    if out_dir.exists() and any(out_dir.iterdir()):
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(out_dir)
    return out_dir


def list_images(d):
    d = Path(d)
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXT) if d.is_dir() else []


def write_label(path, lines):
    Path(path).write_text("".join(l + "\n" for l in lines))


def coco_to_yolo_lines(img_w, img_h, anns, catid2name, name_map):
    """Anotasi COCO (x, y, w, h piksel) -> baris YOLO (kelas xc yc w h, 0-1). Kotak dipotong ke batas gambar."""
    lines = []
    for a in anns:
        tgt = name_map.get(norm(catid2name.get(a.get("category_id"), "")))
        if tgt is None:
            continue
        x, y, w, h = a["bbox"]
        x1, y1 = max(0.0, x), max(0.0, y)
        x2, y2 = min(float(img_w), x + w), min(float(img_h), y + h)
        bw, bh = x2 - x1, y2 - y1
        if bw < 2 or bh < 2:
            continue
        lines.append(f"{NAME2ID[tgt]} {(x1 + x2) / 2 / img_w:.6f} {(y1 + y2) / 2 / img_h:.6f} {bw / img_w:.6f} {bh / img_h:.6f}")
    return lines


def coco_filenames(src_dir):
    cands = sorted(Path(src_dir).rglob("_annotations.coco.json"))
    assert cands, f"_annotations.coco.json tidak ditemukan di {src_dir}"
    data = json.loads(cands[0].read_text())
    return sorted(im["file_name"] for im in data["images"])


def convert_ppes(src_dir, img_out, lbl_out, prefix, keep=None):
    """PPEs (COCO) -> YOLO. keep: himpunan nama file untuk subset (None = semua)."""
    cands = sorted(Path(src_dir).rglob("_annotations.coco.json"))
    assert cands, f"_annotations.coco.json tidak ditemukan di {src_dir}"
    base = cands[0].parent
    data = json.loads(cands[0].read_text())
    catid2name = {c["id"]: c["name"] for c in data["categories"]}
    dropped = sorted({norm(n) for n in catid2name.values()} - set(PPES_MAP))
    per_img = collections.defaultdict(list)
    for a in data["annotations"]:
        per_img[a["image_id"]].append(a)
    img_out, lbl_out = Path(img_out), Path(lbl_out)
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    stems, boxes = [], 0
    for im in data["images"]:
        if keep is not None and im["file_name"] not in keep:
            continue
        src = base / im["file_name"]
        if not src.exists():
            continue
        lines = coco_to_yolo_lines(im["width"], im["height"], per_img[im["id"]], catid2name, PPES_MAP)
        name = safe_name(f"{prefix}_{im['file_name']}")
        shutil.copy2(src, img_out / name)
        write_label(lbl_out / (Path(name).stem + ".txt"), lines)
        stems.append(Path(name).stem)
        boxes += len(lines)
    print(f"  {prefix}: {len(stems)} gambar, {boxes} kotak; kategori dibuang: {dropped or '-'}")
    if stems and boxes == 0:
        raise RuntimeError("Tidak ada kotak yang cocok dengan PPES_MAP; cek nama kategori di _annotations.coco.json")
    return stems


def find_yolo_root(base):
    for d in sorted(Path(base).rglob("train")):
        if d.parent.name == "images" and (d.parent.parent / "labels" / "train").is_dir():
            return d.parent.parent
    raise FileNotFoundError(f"struktur images/train + labels/train tidak ditemukan di {base}")


def copy_cppe(root, split, img_out, lbl_out, repeat=1):
    """Salin Construction-PPE (ID kelas sudah sama dengan CLASSES[:12]); repeat = jumlah salinan."""
    root = Path(root)
    img_out, lbl_out = Path(img_out), Path(lbl_out)
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in list_images(root / "images" / split):
        lab = root / "labels" / split / (p.stem + ".txt")
        text = lab.read_text() if lab.exists() else ""
        for ln in text.splitlines():
            if ln.strip() and not 0 <= int(float(ln.split()[0])) < N_CPPE:
                raise ValueError(f"ID kelas di luar 0-{N_CPPE - 1} pada {lab.name}: bukan arsip Construction-PPE Nov 2026?")
        if text and not text.endswith("\n"):
            text += "\n"
        stem = safe_name(p.stem)
        for k in range(repeat):
            tag = "" if k == 0 else f"_x{k}"
            shutil.copy2(p, img_out / f"cppe_{stem}{tag}{p.suffix.lower()}")
            (lbl_out / f"cppe_{stem}{tag}.txt").write_text(text)
        n += 1
    return n


def complete_labels(model_path, img_dir, lbl_dir, name_map, conf, classes=None, batch=32):
    """Tambahkan deteksi model untuk kelas yang tidak dilabeli sumber. name_map: nama kelas model (dinormalkan) -> kelas akhir."""
    from ultralytics import YOLO
    model = YOLO(str(model_path))
    added = collections.Counter()
    for r in model.predict(source=str(img_dir), conf=conf, imgsz=IMGSZ, batch=batch, classes=classes,
                           stream=True, verbose=False, device=0):
        if r.boxes is None or len(r.boxes) == 0:
            continue
        lines = []
        for c, (x, y, w, h) in zip(r.boxes.cls.tolist(), r.boxes.xywhn.tolist()):
            tgt = name_map.get(norm(r.names[int(c)]))
            if tgt is None:
                continue
            lines.append(f"{NAME2ID[tgt]} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
            added[tgt] += 1
        if lines:
            with open(Path(lbl_dir) / (Path(r.path).stem + ".txt"), "a") as f:
                f.write("".join(l + "\n" for l in lines))
    return added


def count_instances(lbl_dir):
    c = collections.Counter()
    for f in Path(lbl_dir).glob("*.txt"):
        for ln in f.read_text().splitlines():
            if ln.strip():
                c[CLASSES[int(float(ln.split()[0]))]] += 1
    return c


def write_yaml(path, root, train, val, names=None):
    names = names or CLASSES
    fmt = lambda x: ("[" + ", ".join(x) + "]") if isinstance(x, (list, tuple)) else x
    lines = [f"path: {root}", f"train: {fmt(train)}", f"val: {fmt(val)}", "names:"]
    lines += [f"  {i}: {n}" for i, n in enumerate(names)]
    Path(path).write_text("\n".join(lines) + "\n")


def eval_per_class(weights, data_yaml):
    """Validasi satu model; kembalikan {kelas: {p, r, ap50, ap}}, mAP50, mAP50-95 (hanya kelas yang punya contoh)."""
    from ultralytics import YOLO
    model = YOLO(str(weights))
    m = model.val(data=str(data_yaml), imgsz=IMGSZ, batch=BATCH, conf=0.001, iou=0.6, plots=False, verbose=False)
    b = m.box
    out = {}
    for i, c in enumerate(b.ap_class_index):
        try:
            p, r, ap50, ap = b.class_result(i)
        except AttributeError:  # cadangan bila metode ini berganti nama
            p, r, ap50, ap = b.p[i], b.r[i], b.ap50[i], b.ap[i]
        out[model.names[int(c)]] = {"p": float(p), "r": float(r), "ap50": float(ap50), "ap": float(ap)}
    return out, float(b.map50), float(b.map)

## Data

Sel 5 mengunduh sekitar 2,3 GB (Construction-PPE dari GitHub Ultralytics, PPEs dari Hugging Face) dan mengambil `best.pt` baseline dari repo GitHub kamu. Sel 6 sampai 8 menggabungkan, melengkapi label, dan mencetak jumlah objek per kelas.

In [4]:
# 5) Unduh data dan ambil model baseline
cppe_zip = download(CPPE_URL, WORK / 'construction-ppe.zip')
cppe_raw = unzip(cppe_zip, WORK / 'raw_cppe')

ppes_raw = {}
for split, fname in {'train': 'train.zip', 'valid': 'valid.zip', 'test': 'test.zip'}.items():
    z = download(HF_BASE + fname, WORK / f'ppes_{fname}')
    ppes_raw[split] = unzip(z, WORK / f'raw_ppes_{split}')

if not BASELINE_WEIGHTS:
    if COLAB:
        repo_dir = Path('/content/safework')
        if not repo_dir.exists():
            subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_dir)], check=True)
    else:
        repo_dir = REPO
    found = sorted(repo_dir.rglob('best.pt'), key=lambda p: ('ppe-v1' not in str(p), str(p)))
    BASELINE_WEIGHTS = str(found[0]) if found else None

assert BASELINE_WEIGHTS and Path(BASELINE_WEIGHTS).exists() and Path(BASELINE_WEIGHTS).stat().st_size > 1_000_000, \
    'best.pt baseline tidak ditemukan. Jalankan dari repo atau unggah best.pt ke Colab.'
print('baseline:', BASELINE_WEIGHTS)


selesai: construction-ppe.zip (648 MB)
selesai: ppes_train.zip (904 MB)
selesai: ppes_valid.zip (1126 MB)
selesai: ppes_test.zip (245 MB)
baseline: /content/safework/runs/ppe-v1/weights/best.pt


In [5]:
# 6) Gabungkan ke satu dataset YOLO 14 kelas (dibangun ulang dari nol tiap dijalankan)
MERGED = WORK / "ppe_v2"
shutil.rmtree(MERGED, ignore_errors=True)
IMG, LBL = MERGED / "images", MERGED / "labels"

# 6a) Construction-PPE: label paling bersih, diulang CPPE_OVERSAMPLE kali di data latih
cppe_root = find_yolo_root(cppe_raw)
n_all = sum(len(list_images(cppe_root / "images" / s)) for s in ("train", "val", "test"))
if n_all == 1416 or (cppe_root / "images" / "test").is_dir():
    raise RuntimeError("Arsip Construction-PPE ini versi lama (1.416 gambar, 11 kelas). Hapus "
                       "/content/work/construction-ppe.zip dan /content/work/raw_cppe, lalu jalankan ulang dari sel 5.")
n_tr = copy_cppe(cppe_root, "train", IMG / "train_cppe", LBL / "train_cppe", repeat=CPPE_OVERSAMPLE)
n_va = copy_cppe(cppe_root, "val", IMG / "val_cppe", LBL / "val_cppe")
print(f"Construction-PPE: train {n_tr} (x{CPPE_OVERSAMPLE}), val {n_va}")
if (n_tr, n_va) != (800, 200):
    print("PERINGATAN: jumlah gambar bukan 800/200 seperti rilis 3 Nov 2026; pastikan versi dan urutan kelasnya benar.")

# 6b) PPEs (COCO -> YOLO). valid dipecah: sebagian untuk validasi saat latihan, sisanya opsional ikut latih
val_files = coco_filenames(ppes_raw["valid"])
val_pick = set(random.Random(SEED).sample(val_files, min(PPES_VAL_SUBSET, len(val_files))))
n_train = len(convert_ppes(ppes_raw["train"], IMG / "train_ppes", LBL / "train_ppes", "ppes_train"))
n_val = len(convert_ppes(ppes_raw["valid"], IMG / "val_ppes", LBL / "val_ppes", "ppes_val", keep=val_pick))
n_test = len(convert_ppes(ppes_raw["test"], IMG / "test_ppes", LBL / "test_ppes", "ppes_test"))
n_extra = 0
if PPES_VALID_TO_TRAIN:
    rest = set(val_files) - val_pick
    n_extra = len(convert_ppes(ppes_raw["valid"], IMG / "train_ppes", LBL / "train_ppes", "ppes_valid", keep=rest))
print(f"PPEs: train {n_train + n_extra}, val {n_val}, test {n_test}")

Construction-PPE: train 800 (x3), val 200
  ppes_train: 6473 gambar, 13614 kotak; kategori dibuang: -
  ppes_val: 600 gambar, 1302 kotak; kategori dibuang: -
  ppes_test: 1935 gambar, 2884 kotak; kategori dibuang: -
PPEs: train 6473, val 600, test 1935


In [6]:
# 7) Lengkapi label yang tidak ada di PPEs: Person (yolo11m COCO), vest/no_vest/forklift (baseline ppe-v1).
#    Tanpa ini model belajar bahwa orang dan rompi di gambar PPEs adalah latar belakang.
total = collections.Counter()
for split in ("train_ppes", "val_ppes", "test_ppes"):
    if not list_images(IMG / split):
        continue
    a = complete_labels("yolo11m.pt", IMG / split, LBL / split, {"person": "Person"}, PSEUDO_CONF, classes=[0])
    b = complete_labels(BASELINE_WEIGHTS, IMG / split, LBL / split,
                        {"vest": "vest", "no_vest": "no_vest", "forklift": "forklift"}, PSEUDO_CONF)
    print(split, dict(a + b))
    total.update(a + b)

train_ppes {'Person': 18610, 'no_vest': 9726, 'vest': 345, 'forklift': 25}
val_ppes {'Person': 1341, 'no_vest': 717, 'vest': 35, 'forklift': 4}
test_ppes {'Person': 3803, 'no_vest': 2668, 'vest': 192, 'forklift': 37}


In [7]:
# 8) Berkas konfigurasi dan ringkasan jumlah objek per kelas
write_yaml(MERGED / "ppe_v2.yaml", MERGED, ["images/train_cppe", "images/train_ppes"], ["images/val_cppe", "images/val_ppes"])
write_yaml(MERGED / "eval_cppe.yaml", MERGED, "images/train_cppe", "images/val_cppe")
write_yaml(MERGED / "eval_ppes.yaml", MERGED, "images/train_cppe", "images/test_ppes")
write_yaml(MERGED / "baseline_cppe.yaml", MERGED, "images/train_cppe", "images/val_cppe", names=CLASSES[:N_CPPE])

rows = {s: count_instances(LBL / s) for s in ("train_cppe", "train_ppes", "val_cppe", "val_ppes", "test_ppes")}
print(f"{'kelas':<11}" + "".join(f"{s:>12}" for s in rows))
for c in CLASSES:
    print(f"{c:<11}" + "".join(f"{rows[s][c]:>12}" for s in rows))
low = [c for c in CLASSES if rows["train_cppe"][c] + rows["train_ppes"][c] < 150]
if low:
    print("PERINGATAN: contoh latih sedikit (<150) untuk:", low)

kelas        train_cppe  train_ppes    val_cppe    val_ppes   test_ppes
helmet             3846           0         294          42        1240
gloves             4644        3679         381         178           0
vest               4428         345         340          35         192
boots              6009          14         481          99         102
goggles            1836        2692         139         202         300
Person             5886       18610         481        1341        3803
no_helmet          1902           0         179          31        1067
no_goggle           795        2631          82         238          77
no_gloves          2517        4578         203         279           0
no_boots           1707          20         178          89          61
forklift            825          25          73           4          37
no_vest            1311        9726         130         717        2668
mask                  0           0           0          41     

In [8]:
# 9) Latih; Colab memakai GPU 0, .venv Windows memakai CPU bila tidak ada CUDA
from ultralytics import YOLO

for m in MODELS:
    name = f'ppe-v2-{Path(m).stem}'
    wdir = RUNS / name / 'weights'
    if (wdir / 'last.pt').exists():
        print('Ada checkpoint, mencoba melanjutkan:', wdir / 'last.pt')
        try:
            YOLO(str(wdir / 'last.pt')).train(resume=True)
            continue
        except Exception as e:
            print('Tidak bisa dilanjutkan:', e)
            if (wdir / 'best.pt').exists():
                continue

    YOLO(m).train(
        data=str(MERGED / 'ppe_v2.yaml'), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        patience=PATIENCE, workers=2, cache=CACHE, seed=SEED, device=DEVICE,
        close_mosaic=10, save_period=5, project=str(RUNS), name=name,
        exist_ok=True, plots=True, **EXTRA_TRAIN_ARGS,
    )


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/work/ppe_v2/ppe_v2.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ppe-v2-yolo11n, nbs=64, nms=None, opset=None, optimize=Fa

In [9]:
# 10) Uji lolos atau tidak: bandingkan dengan baseline pada 200 gambar validasi Construction-PPE yang sama
def fmt(d, k):
    return f"{d[k]:.2f}" if d else "-"


base, base_map50, _ = eval_per_class(BASELINE_WEIGHTS, MERGED / "baseline_cppe.yaml")
print(f"baseline ppe-v1: mAP50 di Construction-PPE val = {base_map50:.3f}")

for m in MODELS:
    name = f"ppe-v2-{Path(m).stem}"
    best = RUNS / name / "weights" / "best.pt"
    if not best.exists():
        print("best.pt belum ada untuk", name)
        continue
    new_c, map_c, _ = eval_per_class(best, MERGED / "eval_cppe.yaml")
    new_p, map_p, _ = eval_per_class(best, MERGED / "eval_ppes.yaml")
    print(f"\n== {name}: mAP50 Construction-PPE val {map_c:.3f} (baseline {base_map50:.3f}) | PPEs test {map_p:.3f} ==")
    print(f"{'kelas':<11}{'base AP50':>10}{'v2 AP50':>9}{'v2 recall':>10}{'PPEs AP50':>11}{'recall':>8}")
    for c in CLASSES:
        b, n, t = base.get(c), new_c.get(c), new_p.get(c)
        print(f"{c:<11}{fmt(b, 'ap50'):>10}{fmt(n, 'ap50'):>9}{fmt(n, 'r'):>10}{fmt(t, 'ap50'):>11}{fmt(t, 'r'):>8}")
    turun = [c for c, b in base.items() if b["ap50"] >= 0.3 and c in new_c and new_c[c]["ap50"] < b["ap50"] - 0.05]
    if turun:
        print(f"\nPERHATIAN: AP50 turun lebih dari 0,05 pada {turun}. Naikkan CPPE_OVERSAMPLE atau fine-tune sebentar di Construction-PPE saja.")
    else:
        print("\nLOLOS: tidak ada kelas lama yang turun lebih dari 0,05 AP50 dibanding baseline.")
    print("Catatan: pada kolom PPEs, baris Person, vest, no_vest, forklift memakai label otomatis; jangan dipakai sebagai ukuran.")

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,599,724 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 140.6±40.9 MB/s, size: 509.8 KB)
val: Scanning /content/work/ppe_v2/labels/val_cppe... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 491.3it/s 0.4s
val: New cache created: /content/work/ppe_v2/labels/val_cppe.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.3it/s 5.3s
                   all        200       2961      0.871      0.709       0.77      0.544
Speed: 1.3ms preprocess, 4.2ms inference, 0.0ms loss, 2.2ms postprocess per image
baseline ppe-v1: mAP50 di Construction-PPE val = 0.770
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,600,210 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 

In [10]:
# 11) Simpan hasil langsung ke repository atau Google Drive
from ultralytics import YOLO

if COLAB:
    out = Path('/content/out')
else:
    out = REPO / 'runs' / 'ppe-v2' / 'export'
out.mkdir(parents=True, exist_ok=True)

for m in MODELS:
    name = f'ppe-v2-{Path(m).stem}'
    best = RUNS / name / 'weights' / 'best.pt'
    if not best.exists():
        continue
    shutil.copy2(best, out / f'{name}.pt')
    if COLAB and DRIVE_DIR:
        shutil.copy2(best, DRIVE_DIR / f'{name}.pt')
    print('tersimpan:', out / f'{name}.pt', '| kelas:', list(YOLO(str(best)).names.values()))

if COLAB:
    try:
        from google.colab import files
        for f in sorted(out.glob('*.pt')):
            files.download(str(f))
    except Exception as e:
        print('Unduh otomatis tidak tersedia:', e)


tersimpan: /content/out/ppe-v2-yolo11n.pt | kelas: ['helmet', 'gloves', 'vest', 'boots', 'goggles', 'Person', 'no_helmet', 'no_goggle', 'no_gloves', 'no_boots', 'forklift', 'no_vest', 'mask', 'no_mask']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Setelah selesai

- Hasil lokal `.venv`: `runs/ppe-v2/export/ppe-v2-yolo11n.pt`
- Salin atau ganti nama hasil ke: `runs/ppe-v2/weights/best.pt`
- Cocokkan daftar kelas yang dicetak sel 10 dengan `ALIASES` di `safework/labels.py`.
- Jalankan dari .venv:

```powershell
.\.venv\Scripts\Activate.ps1
python -m safework --weights runs\ppe-v2\weights\best.pt --debug
```
